In [36]:
# Install Triton Client and DALI
# !pip install nvidia-dali-cuda120  # Matches Colab's CUDA 12.x
# !pip install tritonclient[all]
# !pip install clip
# !pip install ftfy regex tqdm
# !pip install git+https://github.com/openai/CLIP.git
# !pip install --upgrade tensorrt

## package generally not available in Colab
# !pip install onnxscript onnx
# !pip install -U nvidia-pytriton tritonclient[all]

# !pip install --extra-index-url https://pypi.nvidia.com nvidia-dali-cuda120 

# Download the specific tritonserver version for your A100 environment
# !wget https://github.com/triton-inference-server/server/releases/download/v2.42.0/tritonserver2.42.0-ubuntu22.04.tgz
# !mkdir /content/drive/MyDrive/tmp/triton_server 
!wget -P /content/drive/MyDrive/tmp/ https://github.com/triton-inference-server/server/releases/download/v2.42.0/v2.42.0_ubuntu2204.clients.tar.gz
# !tar -xzf /content/drive/MyDrive/tmp/tritonserver2.41.0-igpu.tar.gz -C /content/drive/MyDrive/tmp/triton_server/ 

--2026-03-04 07:04:33--  https://github.com/triton-inference-server/server/releases/download/v2.42.0/v2.42.0_ubuntu2204.clients.tar.gz
Resolving github.com (github.com)... 20.205.243.166
Connecting to github.com (github.com)|20.205.243.166|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/151636194/6a9779bb-e56d-471f-abb3-15d14ba2dc9f?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-03-04T08%3A00%3A37Z&rscd=attachment%3B+filename%3Dv2.42.0_ubuntu2204.clients.tar.gz&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-03-04T06%3A59%3A42Z&ske=2026-03-04T08%3A00%3A37Z&sks=b&skv=2018-11-09&sig=3rlKsyoteVE1eFFuLiMgkJPRCrQTB5fsq%2F4GOB8srEY%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc3MjYxMTQ3MywibmJmIjoxNzcy

In [28]:
!ls /content/drive/MyDrive/tmp

triton_server  tritonserver2.41.0-igpu.tar.gz


In [ ]:
# trtexec - TensorRT compiler for converting onnx to complied model, .
!apt-get install -y libnvinfer-bin

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Triton DALI model directory structure
# !mkdir -p model_repository/preprocess/1
# mkdir -p model_repository/clip_vision/1
# mkdir -p model_repository/openclip_ensemble/1

In [3]:
# !ln -s /content/drive/Othercomputers/'My Laptop' /data

!ls /content/drive/Othercomputers/'My Laptop'/

!export data_dir=/content/drive/Othercomputers/'My Laptop'/ 

!$data_dir

open-clip


In [4]:
data_dir='/content/drive/Othercomputers/My Laptop' 

In [5]:
# Create a symlink named 'myshortcut' that points to '/content/actual_folder'

!ls -ltr '/content/drive/Othercomputers/My Laptop/open-clip/pipeline'
!ln -s '/content/drive/Othercomputers/My Laptop/open-clip/pipeline' /data

!ls -ltr /data/

total 8
drwx------ 5 root root 4096 Mar  1 09:30 model_repository
drwx------ 2 root root 4096 Mar  2 10:01 models
total 8
drwx------ 5 root root 4096 Mar  1 09:30 model_repository
drwx------ 2 root root 4096 Mar  2 10:01 models


In [ ]:
# Triton DALI model directory structure
# !mkdir -p /data/open-clip/pipeline/model_repository/dali_preprocess/1/
# !mkdir -p  /data/open-clip/pipeline/model_repository/clip_vision/1/
# !mkdir -p  /data/open-clip/pipeline/model_repository/clip_ensemble/1/

In [12]:
!ls /data/models/

model.plan  openai_clip_vision.onnx  openai_clip_vision.onnx.data


In [2]:
import torch
from transformers import CLIPModel
# 1. Download the original OpenAI model
# 'ViT-B-32' refers to the architecture
# 'openai' tells open_clip to fetch the original weights from OpenAI's servers

model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
model.eval()

# Use 77 for the dummy input
# 3. Create dummy input (Standard CLIP resolution is 224x224)
# Batch size 1 for the trace, but we will make it dynamic for Triton, 2 is for dynamic batch size, this was based on multiple trial and error, often the trtexec fails due hardcoded shapes duraing conversation, this ist o make sure that `2` is read `dynamic`
batch_size = 2
seq_len = 77
dummy_input_ids = torch.ones(batch_size, seq_len, dtype=torch.long)
dummy_pixel_values = torch.randn(batch_size, 3, 224, 224)
dummy_attention_mask = torch.ones(batch_size, seq_len, dtype=torch.long)
# 4. Export to ONNX
# Note: Using opset_version 14 for better compatibility with TensorRT 8.x+
# this is the magic porttion Opset : Operator Set - basically hardware specific instructions and optimsation by the providers. 
# For A100/H100 the opset is 17, it fuses `LayerNormalization` & `GroupNormalization` as single operators (fusted kernels): Dont forget FP8 :) for H100
# Also same chagned to 17 due to LayerNorm node does not exist, which is optimized in 17.
# Dynamo=False, this is Torch.dynamo trying to detect dynamic shapes and we provided 77 as fixed
# for B200  : TensorRT 10.x is recommended :: Dual-die Architectre : More on it Later!

# ONNX files use Google’s Protobuf format to store everything (architecture + weights). 
# Protobuf has a hard limit of 2GB for a single file.
#   If your model is < 2GB: Everything is packed into one tidy .onnx file.
#   If your model is > 2GB: The exporter must strip the weights out and 
#   put them into a separate .data file (or multiple files)
#   so the main .onnx file stays under the limit.commended :: Dual-die Architectre : More on it Later!

torch.onnx.export(
    model,
    (dummy_input_ids, dummy_pixel_values, dummy_attention_mask),
    "/data/models/openai_clip.onnx",
    export_params=True,
    opset_version=17,
    do_constant_folding=True,
    input_names=['input_ids', 'pixel_values', 'attention_mask'],
    output_names=['logits_per_image', 'logits_per_text', 'text_embeds', 'image_embeds'],
    # Only make the batch_size dynamic to avoid the conflict
    dynamic_axes={
        'input_ids': {0: 'batch_size'},
        'pixel_values': {0: 'batch_size'},
        'attention_mask': {0: 'batch_size'}
    }
)

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/tmp/ipykernel_31469/1087784032.py:33: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0303 10:57:08.652000 31469 torch/onnx/_internal/exporter/_compat.py:125] Setting ONNX exporter to use operator set version 18 because the requested opset_version 17 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested vers

[torch.onnx] Obtain model graph for `CLIPModel([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `CLIPModel([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...


/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...


[torch.onnx] Translate the graph into ONNX... ✅
Applied 107 of general pattern rewrite rules.


ONNXProgram(
    model=
        <
            ir_version=10,
            opset_imports={'': 17},
            producer_name='pytorch',
            producer_version='2.10.0+cu128',
            domain=None,
            model_version=None,
        >
        graph(
            name=main_graph,
            inputs=(
                %"input_ids"<INT64,[batch_size,77]>,
                %"pixel_values"<FLOAT,[batch_size,3,224,224]>,
                %"attention_mask"<INT64,[batch_size,77]>
            ),
            outputs=(
                %"logits_per_image"<FLOAT,[batch_size,batch_size]>,
                %"logits_per_text"<FLOAT,[batch_size,batch_size]>,
                %"text_embeds"<FLOAT,[batch_size,512]>,
                %"image_embeds"<FLOAT,[batch_size,512]>,
                %"layer_norm_50"<FLOAT,[batch_size,77,512]>,
                %"index_1"<FLOAT,[unk__613,512]>,
                %"add_1197"<FLOAT,[batch_size,50,768]>,
                %"layer_norm_25"<FLOAT,[batch_size,768]>
       

In [ ]:
# trtexec - TensorRT compiler for converting onnx to complied model, .
!apt-get install -y libnvinfer-bin

In [ ]:
# Onnx simplifier package simplifies, what I learned is converting to onnx, certain operation can hardcode 
# shape and reshape operation can hardcode
# !pip install onnx-simplifier

In [ ]:
!pip install onnxsim

In [33]:
import onnx; print([(i.name, i.type.tensor_type.shape.dim[0]) for i in onnx.load("/data/models/openai_clip_dynamic.onnx").graph.input])

[('input_ids', dim_param: "batch_size"
), ('pixel_values', dim_param: "batch_size"
), ('attention_mask', dim_param: "batch_size"
)]


### Simplify the model

In [39]:
import onnx
from onnxsim import simplify

# Load your model
model = onnx.load("/data/models/openai_clip.onnx")

# Use None for the dynamic batch dimension
# This avoids the 'int()' conversion error entirely
input_shapes = {
    "input_ids": [1, 77],
    "pixel_values": [1, 3, 224, 224],
    "attention_mask": [1, 77]
}

# The 'dynamic_input_shape=True' flag is the modern way to do this
model_simp, check = simplify(model, input_shapes=input_shapes, dynamic_input_shape=True)

if check:
    onnx.save(model_simp, "/data/models/openai_clip_sim.onnx")
    print("✅ Success! Dynamic axes preserved.")

WARNING: The argument `dynamic_input_shape=True` is not needed any more, onnxsim can now support dynamic input 
shapes natively, please refer to the latest documentation. An error will be raised in the future.

WARNING: The argument `input_shapes` is deprecated. Please use `overwrite_input_shapes` and/or `test_input_shapes` 
instead. An error will be raised in the future.

✅ Success! Dynamic axes preserved.


### Dynamic Conversion

Most likely this step is not required, if the above step is not run. I got several errors, now have debug this further.

In [40]:
import onnx

# 1. Load the model that currently has static '1' shapes
model_path = "/data/models/openai_clip_sim.onnx"
model = onnx.load(model_path)

# 2. Define the inputs we need to fix
# We want to change the first dimension from '1' to 'batch_size'
dynamic_inputs = ["input_ids", "pixel_values", "attention_mask"]

for inp in model.graph.input:
    if inp.name in dynamic_inputs:
        # dim[0] is the batch dimension
        dim0 = inp.type.tensor_type.shape.dim[0]
        # We overwrite the fixed value with a symbolic name
        dim0.ClearField("dim_value")
        dim0.dim_param = "batch_size"
        print(f"Fixed {inp.name}: Dim 0 is now '{dim0.dim_param}'")

# 3. Save the truly dynamic model
fixed_model_path = "/data/models/openai_clip_dynamic.onnx"
onnx.save(model, fixed_model_path)
print(f"✅ Saved dynamic model to: {fixed_model_path}")

Fixed input_ids: Dim 0 is now 'batch_size'
Fixed pixel_values: Dim 0 is now 'batch_size'
Fixed attention_mask: Dim 0 is now 'batch_size'
✅ Saved dynamic model to: /data/models/openai_clip_dynamic.onnx


### Crating TensorRT engine

In [43]:
# optShapes : maximmize the performance for batch size 32,rest is self explanatory
# memPoolSize : this is to ensure to keep the memory to 2GB,  essentially used by trtexec to test different optimization tactics for each layer. it allocates this memory to run timing test to see which math operation is fastere. At the inference time it will pick 2GB over and above model wights memory and run with it.

!trtexec --onnx=/data/models/openai_clip.onnx \
        --saveEngine=/data/models/model.plan \
        --fp16 \
        --minShapes=pixel_values:1x3x224x224,input_ids:1x77,attention_mask:1x77 \
        --optShapes=pixel_values:32x3x224x224,input_ids:32x77,attention_mask:32x77 \
        --maxShapes=pixel_values:64x3x224x224,input_ids:64x77,attention_mask:64x77 \
        --memPoolSize=workspace:4096MiB \
        --verbose

&&&& RUNNING TensorRT.trtexec [TensorRT v101501] [b29] # trtexec --onnx=/data/models/openai_clip.onnx --saveEngine=/data/models/model.plan --fp16 --minShapes=pixel_values:1x3x224x224,input_ids:1x77,attention_mask:1x77 --optShapes=pixel_values:32x3x224x224,input_ids:32x77,attention_mask:32x77 --maxShapes=pixel_values:64x3x224x224,input_ids:64x77,attention_mask:64x77 --memPoolSize=workspace:4096MiB --verbose
[03/03/2026-12:30:03] [W] Weakly-typed networks have been deprecated in TensorRT. You can use the AutoCast tool (https://nvidia.github.io/TensorRT-Model-Optimizer/guides/8_autocast.html) to convert the network to be strongly typed.
[03/03/2026-12:30:03] [I] === Model Options ===
[03/03/2026-12:30:03] [I] Format: ONNX
[03/03/2026-12:30:03] [I] Model: /data/models/openai_clip.onnx
[03/03/2026-12:30:03] [I] Output:
[03/03/2026-12:30:03] [I] === Build Options ===
[03/03/2026-12:30:03] [I] Memory Pools: workspace: 0.00390625 MiB, dlaSRAM: default, dlaLocalDRAM: default, dlaGlobalDRAM: def

In [46]:
#!ls -ltr /data/
!mv /data/dali_preprocess /data/clip_preprocess

mv: cannot stat '/data/dali_preprocess': No such file or directory


#### Some brilliant insights on optimization of flags in above logs:

##### Sparsity: 
This is the famous Sparsity optimisation introduced for A100, Sparsity 2:4, however this works for sparse models, so Trt wont need it [ReadMore](https://developer.nvidia.com/blog/structured-sparsity-in-the-nvidia-ampere-architecture-and-applications-in-search-engines/)
##### Decomponsable Attentions: 
for MHA, FMHA kernel is used, for Open clip it has found a kernel that is fused, and it does not need to decompose the Matrix multiplication and other parts
##### Refit
Having this flag enabled means the weights can be swapped out without recompiling the model, we are working on inferencing so this wont be neccesary

## Other insights from the logs:

Throughput measured : 438.122 qps ***
90th percentile latency : 3.86475 ms
Host to Device : 1.57896 ms
GPU Compute Time 90th percentile(executing the kernels) : 2.27573 ms
Device to Host : 0.0128167 ms
```
02/28/2026-14:14:51] [I] === Performance summary ===
[02/28/2026-14:14:51] [I] Throughput: 438.122 qps ***
[02/28/2026-14:14:51] [I] Latency: min = 3.79248 ms, max = 4.41853 ms, mean = 3.86752 ms, median = 3.85669 ms, percentile(90%) = 3.86475 ms, percentile(95%) = 3.8678 ms, percentile(99%) = 4.41124 ms
[02/28/2026-14:14:51] [I] Enqueue Time: min = 0.450989 ms, max = 0.654297 ms, mean = 0.496265 ms, median = 0.493408 ms, percentile(90%) = 0.525879 ms, percentile(95%) = 0.537476 ms, percentile(99%) = 0.609253 ms
[02/28/2026-14:14:51] [I] H2D Latency: min = 1.57657 ms, max = 1.59998 ms, mean = 1.57896 ms, median = 1.57849 ms, percentile(90%) = 1.58057 ms, percentile(95%) = 1.58301 ms, percentile(99%) = 1.58582 ms
[02/28/2026-14:14:51] [I] GPU Compute Time: min = 2.20361 ms, max = 2.82626 ms, mean = 2.27573 ms, median = 2.26508 ms, percentile(90%) = 2.27234 ms, percentile(95%) = 2.27533 ms, percentile(99%) = 2.8201 ms
[02/28/2026-14:14:51] [I] D2H Latency: min = 0.010498 ms, max = 0.0150146 ms, mean = 0.0128167 ms, median = 0.0128174 ms, percentile(90%) = 0.013916 ms, percentile(95%) = 0.0141602 ms, percentile(99%) = 0.0145264 ms
[02/28/2026-14:14:51] [I] Total Host Walltime: 3.0083 s
[02/28/2026-14:14:51] [I] Total GPU Compute Time: 2.99942 s
[02/28/2026-14:14:51] [W] * GPU compute time is unstable, with coefficient of variance = 3.40103%.
[02/28/2026-14:14:51] [W]   If not already in use, locking GPU clock frequency or adding --useSpinWait may improve the stability.
[02/28/2026-14:14:51] [I] Explanations of the performance metrics are printed in the verbose logs.
[02/28/2026-14:14:51] [V] 
[02/28/2026-14:14:51] [V] === Explanations of the performance metrics ===
[02/28/2026-14:14:51] [V] Total Host Walltime: the host walltime from when the first query (after warmups) is enqueued to when the last query is completed.
[02/28/2026-14:14:51] [V] GPU Compute Time: the GPU latency to execute the kernels for a query.
[02/28/2026-14:14:51] [V] Total GPU Compute Time: the summation of the GPU Compute Time of all the queries. If this is significantly shorter than Total Host Walltime, the GPU may be under-utilized because of host-side overheads or data transfers.
[02/28/2026-14:14:51] [V] Throughput: the observed throughput computed by dividing the number of queries by the Total Host Walltime. If this is significantly lower than the reciprocal of GPU Compute Time, the GPU may be under-utilized because of host-side overheads or data transfers.
[02/28/2026-14:14:51] [V] Enqueue Time: the host latency to enqueue a query. If this is longer than GPU Compute Time, the GPU may be under-utilized.
[02/28/2026-14:14:51] [V] H2D Latency: the latency for host-to-device data transfers for input tensors of a single query.
[02/28/2026-14:14:51] [V] D2H Latency: the latency for device-to-host data transfers for output tensors of a single query.
[02/28/2026-14:14:51] [V] Latency: the summation of H2D Latency, GPU Compute Time, and D2H Latency. This is the latency to infer a single query.



#### Test all parts, sum to follow later :)

##### Step 1: DALI preprocessing pipeline Test

In [10]:
from nvidia.dali import pipeline_def
import nvidia.dali.fn as fn
import nvidia.dali.types as types

## Number of threads is for typically decoders which are CPU based. A100 has JPEG/WebP decoders.


@pipeline_def(batch_size=32, num_threads=4, device_id=0)
def clip_preprocessing_pipeline(device="gpu"):
    # 1. External Source: We feed raw JPEG bytes and pre-tokenized IDs
     
    jpegs = fn.external_source(device="cpu", name="IMAGE_INPUT")
    input_ids = fn.external_source(device="cpu", name="TEXT_IDS")
    attention_mask = fn.external_source(device="cpu", name="ATTN_MASK")

    # 2. Image Branch (GPU accelerated), mixed actuall falls back to CPU for non JPEG images, for JPEG it uses hardware decoders
    # Experimental decoders handles the Nvidia A100's hardware decoder (NVDEC) errors better for corrupted images
    images = fn.experimental.decoders.image(jpegs, device="mixed", output_type=types.RGB)
    
    # CLIP standard: Resize to 224x224 and Center Crop
    images = fn.resize(images, resize_shorter=224, interp_type=types.INTERP_CUBIC)
    # Normalization (CLIP Values: mean=[0.481, 0.457, 0.408], std=[0.268, 0.261, 0.275])
    # Note: DALI expect values scaled 0-255 or 0-1. 
    # Here we normalize directly to the CLIP distribution.
    # CHW - Channel Height Width
    pixel_values = fn.crop_mirror_normalize(
        images, crop=(224, 224),
        dtype=types.FLOAT,
        output_layout="CHW",
        mean=[0.48144531 * 255, 0.4578275 * 255, 0.40821073 * 255],
        std=[0.26862954 * 255, 0.26130258 * 255, 0.27577711 * 255]
    )

    # 3. Text Branch: Move tokenized IDs to GPU
    # Since tokenization is complex (BPE), we do it on CPU and move to GPU here
    input_ids_gpu = input_ids.gpu()
    attention_mask_gpu = attention_mask.gpu()

    return pixel_values, input_ids_gpu, attention_mask_gpu

In [19]:
# Create a dummy image for testing
import cv2
import numpy as np

test_img = np.zeros((300, 400, 3), dtype=np.uint8)
_, encoded = cv2.imencode('.jpg', test_img)
raw_bytes = np.frombuffer(encoded.tobytes(), dtype=np.uint8)

dummy_ids = np.zeros((32, 77), dtype=np.int32)
dummy_mask = np.ones((32, 77), dtype=np.int32)

# Initialize the pipeline
pipe = clip_preprocessing_pipeline()
pipe.build()

# Feed the batch (DALI expects a list of arrays for External Source)
pipe.feed_input("IMAGE_INPUT", [raw_bytes] * 32)
pipe.feed_input("TEXT_IDS", dummy_ids)
pipe.feed_input("ATTN_MASK", dummy_mask)
# Run and Inspect
outputs = pipe.run()
pixels = outputs[0].as_cpu().as_array()
# metadata = outputs[1].as_cpu().as_array()

print(f"Processed Batch Shape: {pixels.shape}")   # (4, 3, 224, 224)
# print(f"Audit Metadata (H, W, C): {metadata[0]}") # [300, 400, 3]1

Processed Batch Shape: (32, 3, 224, 224)


##### Step 2: Toeknizer Test

In [20]:
import numpy as np
from transformers import CLIPTokenizerFast

# --- THE CLASS TO TEST ---
class TritonPythonModel:
    def initialize(self, args=None):
        # High performance Rust-based tokenizer
        self.tokenizer = CLIPTokenizerFast.from_pretrained(
            "openai/clip-vit-base-patch32", 
            use_fast=True
        )

    def execute(self, requests):
        responses = []
        for request in requests:
            # Simulate pb_utils.get_input_tensor_by_name
            # In a real Triton environment, you'd use the pb_utils library
            in_tensor = request.get_input_tensor_by_name("TEXT_INPUT")
            raw_texts = [t.decode('utf-8') for t in in_tensor.tolist()]

            tokens = self.tokenizer(
                raw_texts,
                padding='max_length',
                max_length=77,
                truncation=True,
                return_tensors="np"
            )

            # In a real class, you'd return pb_utils.InferenceResponse
            # For local testing, we return the raw dict
            responses.append({
                "INPUT_IDS": tokens['input_ids'].astype(np.int32),
                "ATTN_MASK": tokens['attention_mask'].astype(np.int32)
            })
        return responses

# --- 2. THE TEST MOCKING LOGIC ---

class MockTensor:
    def __init__(self, data):
        self.data = data
    def tolist(self):
        return self.data

class MockRequest:
    def __init__(self, input_data):
        self.inputs = {"TEXT_INPUT": MockTensor(input_data)}
    def get_input_tensor_by_name(self, name):
        return self.inputs[name]

In [22]:
# 1. Instantiate the class
model = TritonPythonModel()

# 2. Test the 'initialize' method (check if tokenizer loads correctly)
model.initialize()
print("Initialize: SUCCESS")

# 3. Create a Mock Request (mimicking Triton's numpy-bytes format)
test_data = np.array([b"A photo of an A100 GPU", b"A data researcher at work"], dtype=object)
mock_requests = [MockRequest(test_data)]

# 4. Test the 'execute' method
responses = model.execute(mock_requests)

# 5. Verify the results
for resp in responses:
    print(f"IDs Shape: {resp['INPUT_IDS'].shape}")
    print(f"First ID sequence starts with: {resp['INPUT_IDS'][0][:5]}")

assert resp['INPUT_IDS'].shape == (2, 77), "Shape mismatch in Class output!"
print("Execute: SUCCESS")

Initialize: SUCCESS
IDs Shape: (2, 77)
First ID sequence starts with: [49406   320  1125   539   550]
Execute: SUCCESS


#### Testing sum of all parts : Triton Client

In [32]:
!ls /data/model_repository/

clip_ensemble  clip_inference  clip_preprocess	clip_tokenizer


In [ ]:
# 1. Install PyTriton (This downloads the x86_64 2.42 binaries automatically)
!pip install -U nvidia-pytriton

# 2. Use Python to find exactly where those binaries are hiding
import os, site
pytriton_dir = os.path.join(site.getsitepackages()[0], "pytriton/triton")

print(pytriton_dir)

TRITON_BIN = f"{pytriton_dir}/bin/tritonserver"
BACKENDS_DIR = f"{pytriton_dir}/backends"
LIB_PATH = f"{pytriton_dir}/lib"

# 3. Create a clean symlink to avoid long paths
!ln -sf {TRITON_BIN} /content/tritonserver
!chmod +x /content/tritonserver

print(f"✅ Verified: Binary is at /content/tritonserver")

In [ ]:
# 1. Create the folder structure Triton expects
!mkdir -p /content/triton_backends/dali
!mkdir -p /content/triton_backends/python
!mkdir -p /content/triton_backends/tensorrt

# 2. Download the DALI Backend (v0.17.0 is highly compatible with Triton 2.42)
!wget -q -O /content/triton_backends/dali/libtriton_dali.so \
    https://github.com/triton-inference-server/dali_backend/releases/download/v0.17.0/libtriton_dali.so

# 3. Download the Python Backend (Matches Triton 2.42.0)
!wget -q -O /content/triton_backends/python/libtriton_python.so \
    https://github.com/triton-inference-server/python_backend/releases/download/v2.42.0/libtriton_python.so

# 4. Verify they exist
!ls -R /content/triton_backends

In [73]:
!find -name libtriton_dali.so

In [ ]:
import os
import subprocess
import time

# Update this to your Google Drive path
MODEL_REPO = "/data/model_repository/" 

# Crucial: Add the bundled libs to the system path
env = os.environ.copy()
env["LD_LIBRARY_PATH"] = f"{LIB_PATH}:{env.get('LD_LIBRARY_PATH', '')}"

# Start the server in the background
cmd = [
    "/content/triton_server/bin/tritonserver",
    f"--model-repository={MODEL_REPO}",
    f"--backend-directory=/content/triton_backends",
    "--allow-gpu-metrics=false"
]

print("🚀 Launching Triton 2.42.0...")
process = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

# Monitor the logs until it's ready
for _ in range(60):
    line = process.stdout.readline()
    if line:
        print(line.strip())
        if "Started HTTPService at 0.0.0.0:8000" in line:
            print("\n✅ SERVER READY")
            break
    time.sleep(1)

🚀 Launching Triton 2.42.0...


FileNotFoundError: [Errno 2] No such file or directory: '/content/triton_server/bin/tritonserve'

In [ ]:
import numpy as np
import tritonclient.http as httpclient
from PIL import Image
import io

# 1. Initialize the Client
# Use HTTP for simple notebook testing; use gRPC for high-throughput production.
try:
    client = httpclient.InferenceServerClient(url="localhost:8000")
    print(f"Server Live: {client.is_server_live()}")
except Exception as e:
    print(f"Failed to connect to Triton: {e}")

# 2. Prepare the Image Input (Raw Bytes for DALI)
# We send the raw bytes because our DALI pipeline starts with fn.decoders.image
with open("research_sample.jpg", "rb") as f:
    img_raw = np.frombuffer(f.read(), dtype=np.uint8)
    # Shape: [1, total_bytes]
    img_input_data = img_raw.reshape(1, -1)

# 3. Prepare the Text Input (Raw Strings for Tokenizer)
# Triton expects a numpy object array for strings
text_input_data = np.array([["A high-resolution satellite image of a forest"]], dtype=object)

# 4. Define Triton Inputs
inputs = [
    httpclient.InferInput("IMAGE_BYTES", img_input_data.shape, "UINT8"),
    httpclient.InferInput("TEXT_RAW", text_input_data.shape, "BYTES")
]

# Set the data
inputs[0].set_data_from_numpy(img_input_data)
inputs[1].set_data_from_numpy(text_input_data)

# 5. Define Requested Outputs (Including our new Similarity Score)
outputs = [
    httpclient.InferRequestedOutput("IMAGE_EMBEDS"),
    httpclient.InferRequestedOutput("TEXT_EMBEDS"),
    httpclient.InferRequestedOutput("SIMILARITY_SCORE")
]

# 6. RUN THE FULL PIPELINE
print("Sending request to ensemble...")
response = client.infer(model_name="clip_ensemble", inputs=inputs, outputs=outputs)

# 7. Access Results
img_emb = response.as_numpy("IMAGE_EMBEDS")
txt_emb = response.as_numpy("TEXT_EMBEDS")
score = response.as_numpy("SIMILARITY_SCORE")

print("-" * 30)
print(f"Image Embedding Shape: {img_emb.shape}")
print(f"Text Embedding Shape:  {txt_emb.shape}")
print(f"Match Similarity:      {score[0][0]:.4f}")